# Import Research Dataset to MPLAB ML

Learn how to import datasets with existing labels into MPLAB ML:
- Load pre-labeled sensor data from research sources
- Work with IEEE, Kaggle, academic, or your own labeled datasets
- Create segments from continuous labels
- Build .dclproj manifest files
- Upload to MPLAB ML

**Use Case:** You have sensor data with existing labels (from published research, academic studies, or your own R&D) and want to use it in MPLAB ML.

**Example Dataset:** IEEE Arc Fault Detection Dataset (series_load sample)
- 200,000 samples (~12.5 seconds)
- 16 kHz sampling rate
- Labels: -1 (normal), 0 (transient), 1 (arc fault)

---
## 1. Setup

In [ ]:
!pip install mplabml --quiet
print("✓ MPLAB ML SDK installed")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from getpass import getpass
from mplabml import Client

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 4)

print("✓ Libraries imported")

---
## 2. Quick Data Quality Recap

💡 **Remember from understanding-data.ipynb:**
- Data should be in sequential format (one reading per row)
- Check for missing values, outliers, label consistency
- Visualize to understand patterns

This dataset is **already cleaned** - it passed all quality checks:
- ✅ Sequential format
- ✅ No missing values
- ✅ No outliers
- ✅ Consistent labels

Now we'll focus on **converting these labels into a format MPLAB ML understands**.

---
## 3. Load Pre-Labeled Dataset

**We'll try loading from GitHub first (works everywhere), with automatic fallback to local files if needed.**

In [ ]:
# Try loading from GitHub first (recommended - works everywhere)
GITHUB_URL = 'https://raw.githubusercontent.com/MicrochipTech/mplabml-sdk-examples/main/datasets/series_load_sample.csv'

try:
    df = pd.read_csv(GITHUB_URL)
    print("✓ Loaded from GitHub")
except Exception as e:
    print(f"Could not load from GitHub: {e}")
    print("\nTrying local file...")
    
    # Fallback to local file
    # For Colab: Upload to /content/
    # For local Jupyter: Clone repo or use custom path
    
    try:
        df = pd.read_csv('/content/series_load_sample.csv')
        print("✓ Loaded from /content/")
    except:
        try:
            df = pd.read_csv('../datasets/series_load_sample.csv')
            print("✓ Loaded from ../datasets/")
        except:
            raise FileNotFoundError(
                "Could not find dataset. Please either:\n"
                "1. Check your internet connection (for GitHub)\n"
                "2. Upload series_load_sample.csv to /content/ (Colab)\n"
                "3. Clone the repo or specify the correct path"
            )

print(f"\nDataset loaded: {len(df):,} samples")
print(f"Duration: {df['Time'].max():.2f} seconds")
print(f"Sampling rate: ~{1/(df['Time'].iloc[1] - df['Time'].iloc[0]):.0f} Hz")
print(f"Columns: {list(df.columns)}")

In [ ]:
# Inspect the data
print("First 10 rows:")
display(df.head(10))

print("\nDataset info:")
df.info()

In [ ]:
# Label distribution
print("Label distribution:")
label_counts = df['label'].value_counts().sort_index()

label_names = {-1.0: 'Normal', 0.0: 'Transient', 1.0: 'Arc Fault'}
for label, count in label_counts.items():
    pct = (count / len(df)) * 100
    name = label_names.get(label, 'Unknown')
    print(f"  {int(label):2d} ({name:>10s}): {count:>7,} samples ({pct:>5.1f}%)")

print(f"\nTotal: {len(df):,} samples")

---
## 4. Visualize Labels Over Time

Understanding how labels change over time is crucial for creating segments.

In [ ]:
# Plot signal with labels
plt.figure(figsize=(16, 5))

# Color map for labels
colors = {-1.0: 'green', 0.0: 'orange', 1.0: 'red'}
labels_map = {-1.0: 'Normal', 0.0: 'Transient', 1.0: 'Arc Fault'}

# Plot every 10th point for performance
plot_data = df[::10].copy()

for label_val in sorted(df['label'].unique()):
    mask = plot_data['label'] == label_val
    plt.scatter(plot_data[mask]['Time'], 
                plot_data[mask]['current_signal'],
                c=colors[label_val], 
                label=labels_map[label_val],
                alpha=0.6, s=2)

plt.xlabel('Time (seconds)', fontsize=12)
plt.ylabel('Current Signal (A)', fontsize=12)
plt.title('Arc Fault Dataset - Labels Over Time', fontsize=14, fontweight='bold')
plt.legend(loc='upper right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 Notice how labels change as the signal characteristics change")
print("   These transition points will become 'segments' for MPLAB ML")

---
## 5. Understanding Segments

**What is a segment?**
A segment is a continuous section of data with the **same label**.

**Example:**
```
Samples 0-1000:     label = normal    → Segment 1
Samples 1001-1500:  label = transient → Segment 2  
Samples 1501-2000:  label = arc_fault → Segment 3
```

**Why segments?**
MPLAB ML needs to know where each label starts and ends to properly train models.

---
## 6. Create Segments from Labels

Let's convert our continuous label column into discrete segments.

In [ ]:
def create_segments_from_labels(df, label_col='label'):
    """
    Create segments by detecting where labels change.
    
    Parameters:
    -----------
    df : DataFrame
        DataFrame with continuous label column
    label_col : str
        Name of the label column
    
    Returns:
    --------
    list : Segments in MPLAB ML format
    """
    # Map numeric labels to string names
    label_map = {
        -1.0: 'normal',
        0.0: 'transient', 
        1.0: 'arc_observed'
    }
    
    segments = []
    current_label = df[label_col].iloc[0]
    start_idx = 0
    
    # Scan through data to find label changes
    for i in range(1, len(df)):
        if df[label_col].iloc[i] != current_label:
            # Label changed - save previous segment
            segments.append({
                "name": "Label",
                "value": label_map[current_label],
                "start": int(start_idx),
                "end": int(i - 1)
            })
            
            # Start new segment
            current_label = df[label_col].iloc[i]
            start_idx = i
    
    # Add final segment
    segments.append({
        "name": "Label",
        "value": label_map[current_label],
        "start": int(start_idx),
        "end": int(len(df) - 1)
    })
    
    return segments

# Create segments
segments = create_segments_from_labels(df)

print(f"✓ Created {len(segments)} segments\n")
print("Segments:")
for i, seg in enumerate(segments, 1):
    duration = seg['end'] - seg['start'] + 1
    time_start = df['Time'].iloc[seg['start']]
    time_end = df['Time'].iloc[seg['end']]
    print(f"  {i}. {seg['value']:<13s}: samples {seg['start']:>6,} to {seg['end']:>6,} "
          f"({duration:>6,} samples, {time_start:.3f}s to {time_end:.3f}s)")

### Verify Segments

Let's check that our segments match the actual data:

In [ ]:
# Verify each segment
print("Segment verification:")
label_map = {-1.0: 'normal', 0.0: 'transient', 1.0: 'arc_observed'}

for i, seg in enumerate(segments, 1):
    # Get actual labels in this range
    actual_labels = df['label'].iloc[seg['start']:seg['end']+1].unique()
    actual_names = [label_map[l] for l in actual_labels]
    
    # Check if segment has only one label
    if len(actual_labels) == 1 and actual_names[0] == seg['value']:
        print(f"  ✓ Segment {i}: {seg['value']} - Correct!")
    else:
        print(f"  ❌ Segment {i}: Expected {seg['value']}, found {actual_names}")

print("\n✓ All segments verified!")

---
## 7. Prepare Data for Upload

MPLAB ML needs:
1. CSV file with the sensor data
2. .dclproj manifest that links the CSV to segments

In [ ]:
# Create output directory
OUTPUT_DIR = 'research_dataset_import'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✓ Created output directory: {OUTPUT_DIR}")

In [ ]:
# Save CSV (without the label column - MPLAB ML will use segments instead)
csv_filename = 'sensor_data.csv'
csv_path = os.path.join(OUTPUT_DIR, csv_filename)

# Save only Time and current_signal columns
df[['Time', 'current_signal']].to_csv(csv_path, index=False)

print(f"✓ Saved CSV: {csv_path}")
print(f"  Size: {os.path.getsize(csv_path) / (1024*1024):.1f} MB")

---
## 8. Build .dclproj Manifest

The manifest is a JSON file that tells MPLAB ML:
- Which CSV file contains the data
- Metadata about the dataset
- Where each label segment starts and ends

In [ ]:
# Create manifest structure
manifest = [{
    "file_name": csv_filename,
    "metadata": [
        {"name": "example_type", "value": "research_dataset_import"},
        {"name": "use_case", "value": "arc_fault_detection"},
        {"name": "source", "value": "IEEE_series_load_sample"},
        {"name": "sampling_rate_hz", "value": "16000"},
        {"name": "description", "value": "example_import_from_research_dataset"}
    ],
    "sessions": [{
        "session_name": "Session_1",
        "segments": segments
    }]
}]

# Save manifest (.dclproj file)
manifest_filename = 'research_dataset_import.dclproj'
manifest_path = os.path.join(OUTPUT_DIR, manifest_filename)
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Saved manifest: {manifest_path}")
print(f"\nManifest preview (first 500 chars):")
print(json.dumps(manifest, indent=2)[:500] + "...")

### Understanding the Manifest Structure

**Important:** Metadata values can only contain **letters, numbers, and underscores**. 
- ✅ Good: `tutorial_example`, `arc_fault_detection`, `16000`
- ❌ Bad: `Tutorial: Example`, `arc-fault (detection)`, `16,000`

```json
[
  {
    "file_name": "sensor_data.csv",            // Which CSV file
    "metadata": [                              // Dataset information
      {"name": "example_type", "value": "manual_labeling_tutorial"},
      {"name": "use_case", "value": "arc_fault_detection"},
      {"name": "description", "value": "tutorial_example_with_prelabeled_data"},
      ...
    ],
    "sessions": [                              // Can have multiple sessions
      {
        "session_name": "Session_1",
        "segments": [                          // Label definitions
          {
            "name": "Label",
            "value": "normal",                 // Label name (also letters/numbers/underscores only)
            "start": 0,                        // Start sample index
            "end": 50000                       // End sample index
          },
          ...
        ]
      }
    ]
  }
]
```

---
## 9. Verify Project Structure

Before uploading, verify everything is in place:

**Required files:**
1. `sensor_data.csv` - Your sensor data
2. `manual_labeling_tutorial.dclproj` - Manifest with label segments

In [ ]:
print("Project structure:")
print(f"\n{OUTPUT_DIR}/")

for item in os.listdir(OUTPUT_DIR):
    path = os.path.join(OUTPUT_DIR, item)
    if os.path.isfile(path):
        size = os.path.getsize(path) / 1024
        print(f"  ├── {item:<30s} ({size:>8.1f} KB)")

print("\n✓ Project ready for upload!")

---
## 10. Upload to MPLAB ML

Now we'll upload the project to MPLAB ML. The SDK will:
1. Check for and delete any existing project with the same name
2. Upload the CSV file
3. Parse the manifest
4. Apply all segments/labels automatically

**Note:** If a previous upload failed, the cleanup step will remove the partial project before retrying.

In [ ]:
# Authenticate
api_key = getpass("Enter your MPLAB ML API Key: ")
client = Client(api_key=api_key)

print("✓ Connected to MPLAB ML!")

### Optional: Manually Delete Existing Project

**Only run this if you want to manually clean up before uploading:**

In [ ]:
# Uncomment and run ONLY if you want to manually delete the project first

# PROJECT_TO_DELETE = "Research_Dataset_Import_Tutorial"
# try:
#     client.delete_project(PROJECT_TO_DELETE)
#     print(f"✓ Deleted project: {PROJECT_TO_DELETE}")
# except Exception as e:
#     print(f"ℹ️  Could not delete (may not exist): {e}")

In [ ]:
# Upload project
PROJECT_NAME = "Research_Dataset_Import_Tutorial"

print("=" * 70)
print(f"UPLOADING PROJECT: {PROJECT_NAME}")
print("=" * 70)
print(f"\nManifest: {manifest_filename}")
print(f"Upload directory: {OUTPUT_DIR}")

# Check if project already exists and delete it
print(f"\n🔍 Checking for existing project...")
try:
    projects_df = client.list_projects()
    if PROJECT_NAME in projects_df['Name'].values:
        print(f"   ⚠️  Project '{PROJECT_NAME}' already exists")
        print(f"   🗑️  Deleting existing project...")
        client.delete_project(PROJECT_NAME)
        print(f"   ✓ Deleted successfully")
    else:
        print(f"   ✓ No existing project found")
except Exception as e:
    print(f"   ℹ️  Could not check for existing project: {e}")

print(f"\n⏳ Upload in progress...")
print("   This may take 30-60 seconds...\n")

try:
    # Upload project with manifest
    result = client.upload_project_dcli(
        name=PROJECT_NAME,
        dcli_path=manifest_path,
        force=True  # Overwrite if project already exists
    )

    print("\n" + "=" * 70)
    print("✅ UPLOAD COMPLETED SUCCESSFULLY!")
    print("=" * 70)
    print(f"\nProject '{PROJECT_NAME}' is now in MPLAB ML!\n")
    print("💡 This is a tutorial example using arc fault research data.")
    print("   You can safely delete it after exploring the import workflow.\n")
    
except Exception as e:
    print("\n" + "=" * 70)
    print("❌ UPLOAD FAILED")
    print("=" * 70)
    print(f"\nError: {e}\n")
    print("Troubleshooting:")
    print("1. Try deleting the project manually in MPLAB ML web interface")
    print("2. Use a different project name (change PROJECT_NAME above)")
    print("3. Check your internet connection")
    print("4. Verify MPLAB ML credentials (API key)")
    print("5. Ensure manifest and CSV files are in same directory")
    print(f"6. Check files exist in: {OUTPUT_DIR}\n")
    raise

---
## 11. Verify Upload

In [ ]:
# List projects to verify
projects_df = client.list_projects()

if PROJECT_NAME in projects_df['Name'].values:
    print(f"✅ Found project: {PROJECT_NAME}\n")
    project_info = projects_df[projects_df['Name'] == PROJECT_NAME]
    display(project_info[['Name', 'Last Modified', 'Created']])
else:
    print(f"❌ Project not found: {PROJECT_NAME}")

print("\n💡 Next steps:")
print("   1. Go to https://mplabml.microchip.com")
print(f"   2. Open project: {PROJECT_NAME}")
print("   3. View your data with labels already applied!")
print("   4. Create a query to select your training data")
print("   5. Build a pipeline and train a model")

---
## Summary

### What We Accomplished:

1. ✅ **Loaded pre-labeled data** - 200k samples with existing labels
2. ✅ **Understood segments** - Continuous sections with same label
3. ✅ **Created segments** - Detected label transitions automatically
4. ✅ **Built manifest** - .dclproj file linking CSV to segments
5. ✅ **Uploaded to MPLAB ML** - Complete project with labels

### Key Concepts:

**Segment:**
```json
{
  "name": "Label",
  "value": "normal",
  "start": 0,
  "end": 50000
}
```
Tells MPLAB ML: "Samples 0-50000 are labeled 'normal'"

**Note:** Label values must use only letters, numbers, and underscores.

**Manifest (.dclproj):**
- JSON file describing dataset structure
- Links CSV file to label segments
- Required for uploading pre-labeled data

**Upload Process:**
1. CSV + .dclproj in same folder
2. `client.upload_project_dcli(name, dcli_path, force=True)`
3. Labels automatically applied in MPLAB ML

### Workflow for Your Own Data:

```python
# 1. Load your labeled data
df = pd.read_csv('my_data.csv')

# 2. Create segments from labels
segments = create_segments_from_labels(df, label_col='my_label_column')

# 3. Save CSV (without label column)
df.drop('my_label_column', axis=1).to_csv('sensor_data.csv', index=False)

# 4. Build manifest (.dclproj)
manifest = [{
    "file_name": "sensor_data.csv",
    "metadata": [
        {"name": "description", "value": "your_dataset_description"},
        {"name": "use_case", "value": "your_application"},
        {"name": "sampling_rate_hz", "value": "1000"}
        # NOTE: Values can only contain letters, numbers, and underscores!
    ],
    "sessions": [{"session_name": "Session_1", "segments": segments}]
}]

# 5. Save manifest
with open('my_project.dclproj', 'w') as f:
    json.dump(manifest, f, indent=2)

# 6. Upload
client.upload_project_dcli(
    name="My_ML_Project",
    dcli_path="my_project.dclproj",
    force=True
)
```

---

## Next Steps

**Continue Learning:**
- [Auto-Label with Events](./auto-label-with-events.ipynb) - Label using button press, GPIO triggers (coming soon)

**In MPLAB ML:**
1. Create a query to select training data
2. Build feature extraction pipeline
3. Train and evaluate models
4. Generate Knowledge Pack for deployment

---

**🎉 Congratulations!** You've successfully imported research data with labels to MPLAB ML!